Homework 3

Q1(a) and (b): Loading the data, cleaning it and splitting into train and test

In [1]:
import re, glob, os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
RNG_SEED = 552
np.random.seed(RNG_SEED)


DATA_ROOT = '/Users/sharvari/Downloads/AReM'
assert os.path.isdir(DATA_ROOT), f'DATA_ROOT not found: {DATA_ROOT}'
print('folders found:', sorted(os.listdir(DATA_ROOT)))
COLS = ['time','avg_rss12','var_rss12','avg_rss13','var_rss13','avg_rss23','var_rss23']
SERIES = COLS[1:]
ACTIVITIES = ['bending1','bending2','cycling','lying','sitting','standing','walking']

def read_instance(path):
    rows = []
    for line in open(path).read().splitlines():
        if not line.strip() or line.startswith('#'):
            continue
        toks = [t for t in re.split(r'[,\s]+', line.strip()) if t]  
        rows.append([float(t) for t in toks[:7]])
    df = pd.DataFrame(rows, columns=COLS).set_index('time')
    if len(df) != 480:                                              
        df = df.reindex(np.arange(0, 480*250, 250)).interpolate(limit_direction='both')
    df = df.reset_index(drop=True)
    assert df.shape == (480, 6)
    return df

def load_all():
    inst = []
    for act in ACTIVITIES:
        files = sorted(glob.glob(f'{DATA_ROOT}/{act}/dataset*.csv'),
                       key=lambda p: int(re.findall(r'(\d+)', os.path.basename(p))[0]))
        for f in files:
            num = int(re.findall(r'(\d+)', os.path.basename(f))[0])
            limit = 2 if act in ('bending1','bending2') else 3
            inst.append(dict(activity=act, num=num, test=num <= limit, series=read_instance(f)))
    return inst

STATS = ['min','max','mean','median','std','q1','q3']
def stats_of(x):
    return [x.min(), x.max(), x.mean(), np.median(x), x.std(ddof=1), np.percentile(x,25), np.percentile(x,75)]

def feature_names(l=1):
    return [f'{st}{s+1}' + ('' if l == 1 else f'_p{part+1}')
            for part in range(l) for s in range(6) for st in STATS]

def extract(series_df, l=1):
    out = []
    for part in np.array_split(np.arange(len(series_df)), l):
        sub = series_df.iloc[part]
        for c in SERIES:
            out += stats_of(sub[c].values)
    return out

def build_X(instances, l=1):
    return pd.DataFrame([extract(i['series'], l) for i in instances], columns=feature_names(l))

folders found: ['.DS_Store', 'bending1', 'bending2', 'bendingType.pdf', 'cycling', 'lying', 'sensorsPlacement.pdf', 'sitting', 'standing', 'walking']


In [2]:
instances = load_all()
train = [i for i in instances if not i['test']]
test  = [i for i in instances if i['test']]
y_tr_act = np.array([i['activity'] for i in train]); y_te_act = np.array([i['activity'] for i in test])
y_tr = np.array([a.startswith('bending') for a in y_tr_act]).astype(int)   # binary (bending=1)
y_te = np.array([a.startswith('bending') for a in y_te_act]).astype(int)
print(len(instances), len(train), len(test), y_tr.sum(), y_te.sum())

88 69 19 9 4


Q1(c)i: Time-domain features used for time series classification

The usual time-domain features are:

Location: minimum, maximum, mean, median, mode, quartiles, range, mid-range, trimmed mean.
Spread and shape: standard deviation, variance, interquartile range, mean absolute deviation, skewness, kurtosis, coefficient of variation.
Energy and magnitude: root mean square, sum of squares (energy), peak-to-peak amplitude, mean absolute value.
Dynamics: mean and variance of the first differences, number of zero or mean crossings, number of peaks, autocorrelation at a given lag, linear trend slope.
The assignment asks for min, max, mean, median, standard deviation, first quartile and third quartile, so those are the ones I use.


Q1(c)ii: Extracting the features

For each of the 88 instances and each of the 6 series I compute these 7 statistics, which gives 42 features per instance. The table is saved as features_all_instances.csv (columns min1 max1 mean1 median1 std1 q11 q31 ... q16 q36). I use the raw values and do not normalize them.

In [3]:
X_all = build_X(instances, 1)
feat_table = pd.concat([pd.DataFrame({'Instance': range(1, 89), 'activity': [i['activity'] for i in instances],
                                      'dataset': [i['num'] for i in instances], 'is_test': [i['test'] for i in instances]}), X_all], axis=1)
feat_table.to_csv('features_all_instances.csv', index=False)
print(feat_table.shape); print(feat_table.iloc[:5, :12].round(3).to_string())

(88, 46)
   Instance  activity  dataset  is_test   min1   max1   mean1  median1   std1    q11    q31  min2
0         1  bending1        1     True  37.25  45.00  40.625    40.50  1.477  39.25  42.00   0.0
1         2  bending1        2     True  38.00  45.67  42.813    42.50  1.436  42.00  43.67   0.0
2         3  bending1        3    False  35.00  47.40  43.954    44.33  1.559  43.00  45.00   0.0
3         4  bending1        4    False  33.00  47.75  42.180    43.50  3.671  39.15  45.00   0.0
4         5  bending1        5    False  33.00  45.75  41.678    41.75  2.243  41.33  42.75   0.0


Q1(c)iii: 90% bootstrap confidence intervals for the standard deviation of each feature

For every one of the 42 features I resample the 88 instances with replacement 5,000 times, compute the sample standard deviation each time, and take the 5th and 95th percentiles as the 90% interval (percentile bootstrap). The point estimate is the standard deviation on the original data.

Two things I noticed in the table. min2 and min4 are exactly 0 for all instances, so their standard deviation and interval are 0. The widest intervals belong to min5, min1, q15, median5 and mean5, so the features of series 5 (and min1) are the ones that vary most between instances, and also the ones where the estimate of their spread is least certain.

In [4]:
rng = np.random.default_rng(RNG_SEED); B = 5000; A = X_all.values; n = len(A)
idx = rng.integers(0, n, size=(B, n))
boot = np.stack([A[i].std(axis=0, ddof=1) for i in idx])          
boot_tab = pd.DataFrame({'feature': X_all.columns, 'std': A.std(axis=0, ddof=1),
                         'CI90_low': np.percentile(boot, 5, axis=0), 'CI90_high': np.percentile(boot, 95, axis=0),
                         'boot_SE': boot.std(axis=0, ddof=1)})
boot_tab.to_csv('bootstrap_std_ci.csv', index=False)
print(boot_tab.round(3).to_string(index=False))

feature   std  CI90_low  CI90_high  boot_SE
   min1 9.570     8.275     10.768    0.765
   max1 4.394     3.351      5.270    0.587
  mean1 5.336     4.714      5.898    0.358
median1 5.440     4.790      6.009    0.370
   std1 1.772     1.577      1.946    0.114
    q11 6.154     5.575      6.666    0.331
    q31 5.139     4.347      5.851    0.456
   min2 0.000     0.000      0.000    0.000
   max2 5.063     4.618      5.398    0.238
  mean2 1.574     1.393      1.705    0.096
median2 1.412     1.237      1.542    0.094
   std2 0.884     0.803      0.941    0.043
    q12 0.946     0.828      1.035    0.063
    q32 2.125     1.890      2.294    0.124
   min3 2.956     2.765      3.108    0.106
   max3 4.875     4.180      5.464    0.394
  mean3 4.008     3.430      4.504    0.330
median3 4.036     3.446      4.547    0.340
   std3 0.947     0.762      1.114    0.107
    q13 4.221     3.639      4.716    0.331
    q33 4.172     3.552      4.701    0.350
   min4 0.000     0.000      0.0

Q1(c)iv: Choosing the three most important time-domain features

The suggested option was min, mean and max, but I decided not to use it. The minimum of the variance series is almost 0 (min2 and min4 are exactly constant and min6 is below 0.5 everywhere), so it tells us very little there. To decide, I compared the ANOVA F-score of each statistic on the training set, for the binary problem (bending vs. the rest) and for the 7-class problem. Max is the strongest for bending vs. the rest and mean is the strongest for the seven activities. I added the standard deviation as a measure of how spread out the signal is. I should be honest that it is not a perfect choice: its F-score is the lowest for the binary task, and the correlation matrices printed below show that for the variance series (2 and 6) it is highly correlated with the mean (about 0.97–0.98), so there it adds less than I first thought. It is less redundant for series 1. My three selected features are therefore mean, max and standard deviation.

In [5]:
from sklearn.feature_selection import f_classif
Xtr1 = build_X(train, 1)
keep = Xtr1.columns[Xtr1.std() > 0]
Fb = pd.Series(f_classif(Xtr1[keep], y_tr)[0], keep); F7 = pd.Series(f_classif(Xtr1[keep], y_tr_act)[0], keep)
print('mean F (binary) by statistic:\n', Fb.groupby(lambda c: re.sub(r'\d', '', c)).mean().round(1).sort_values(ascending=False).to_string())
print('\nmean F (7-class) by statistic:\n', F7.groupby(lambda c: re.sub(r'\d', '', c)).mean().round(1).sort_values(ascending=False).to_string())
CHOSEN = ['mean', 'max', 'std']
print('\nminima of the variance series (train): min2, min4 are constant; min6 range =',
      (round(Xtr1['min6'].min(), 3), round(Xtr1['min6'].max(), 3)))
for s_ in (1, 2, 6):                                   
    print(f'\ncorrelation of mean/max/std, series {s_}:'); print(Xtr1[[f'mean{s_}', f'max{s_}', f'std{s_}']].corr().round(2).to_string())
print('\nChosen time-domain features:', CHOSEN)

mean F (binary) by statistic:
 max       18.1
mean      10.6
q          9.8
median     9.4
min        8.6
std        3.5

mean F (7-class) by statistic:
 mean      368.3
median    317.7
q         252.3
std       109.2
max        30.0
min        14.6

minima of the variance series (train): min2, min4 are constant; min6 range = (0.0, 0.43)

correlation of mean/max/std, series 1:
       mean1  max1  std1
mean1   1.00  0.55 -0.45
max1    0.55  1.00  0.40
std1   -0.45  0.40  1.00

correlation of mean/max/std, series 2:
       mean2  max2  std2
mean2   1.00  0.89  0.98
max2    0.89  1.00  0.94
std2    0.98  0.94  1.00

correlation of mean/max/std, series 6:
       mean6  max6  std6
mean6   1.00  0.83  0.97
max6    0.83  1.00  0.90
std6    0.97  0.90  1.00

Chosen time-domain features: ['mean', 'max', 'std']


Q2: ISLR 3.7.4

Setup: For (a) and (b), X and Y have a linear true relationship, Y = β0 + β1 X + ε. I fit a linear and a cubic regression.

(a) Training RSS, true relationship linear. The cubic model contains the linear model as a special case, so its training RSS is lower or equal. Extra flexibility never increases the training RSS, and here any improvement comes from fitting noise.

(b) Test RSS, true relationship linear. I expect the linear model to have the lower test RSS. It has the right form (no bias) and lower variance, while the cubic model overfits the training noise (higher variance, no reduction in bias).

(c) Training RSS, true relationship not linear. Again the cubic has lower or equal training RSS, for the same nested-model reason.

(d) Test RSS, true relationship not linear. There is not enough information to be sure. If the truth is closer to cubic than to linear, the cubic has lower test RSS because the linear model has a large bias. If it is only slightly non-linear, the cubic's extra variance can outweigh the bias reduction and the linear model wins. So it depends on how far the true relationship is from linear. If there is clear curvature, I would expect the cubic model to win.

Q3: It was mentioned in the PDF document that Q3 is for extra practice so, even though I have solved it, I did not write answer for it here.


Q4: As it was mentioned in the PDF document Q4 needs to be submitted with homework 4 code. Hence, I have not added answer to the Question 4 in this homework.